# Activity: How People Spend Their Day: Filled

Data Science Training Intensive · NC State University Libraries, Data Science Services

**The data:** 2,000 people's diaries from the American Time Use Survey (ATUS), 2003–2023. It's the same file the DSTI spreadsheets track cleans in Google Sheets, so you can compare answers.

**The question:** How do people split their day between household chores and leisure, by number of children and by work status?

Part 1 uses the plain Python from the intro session. Part 2 does the same and more with Polars, the library from the next workshop.

This takes about 45 minutes. Run the cells from the top, one at a time.

This is the filled notebook: every exercise has an answer, so check your work here. To do the activity yourself, use the [learner notebook](https://colab.research.google.com/github/NCSU-Libraries/dsti-python/blob/main/notebooks/activity-time-use.ipynb).

**Using Colab?** Click **Copy to Drive** first.

Activity page: [https://ncsu-libraries.github.io/dsti-python/activities/time-use.html](https://ncsu-libraries.github.io/dsti-python/activities/time-use.html)

## Get the data

Two files hold the data. `atus_working.csv` is the one you clean. It has mistakes in it on purpose. `atus_raw.csv` has the original survey codes. It's the source of truth.

Source: American Time Use Survey (U.S. Bureau of Labor Statistics), via ICPSR study 36268. The DSTI team sampled 2,000 respondents from diary years 2003–2023.

In [ ]:
# Download the two data files into this folder (skipped if they're already here)
from pathlib import Path
from urllib.request import urlretrieve

BASE = "https://ncsu-libraries.github.io/dsti-python/data/"
for name in ["atus_working.csv", "atus_raw.csv"]:
    if not Path(name).exists():
        urlretrieve(BASE + name, name)
        print("Downloaded", name)

# Part 1: Plain Python

## 1. Read the file

The `csv` module reads a CSV file into a list of dictionaries. There is one dictionary per row, with the column names as keys.

`with` opens the file and closes it again when the indented block ends. (`newline=""` and `encoding="utf-8"` are settings the `csv` module likes.)

In [ ]:
import csv

with open("atus_working.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

In [ ]:
# How many rows are there?
len(rows)

In [ ]:
# Look at the first row
rows[0]

Every value is text, even Age (`'78'`). The `csv` module doesn't guess types.

## 2. Count the values in a column

A dictionary can count things. The keys are the values you've seen, and each value is how many times you've seen it. `value in counts` is True when `value` is already a key.

In [ ]:
# Write count_values(column): return a dictionary of how often each value appears
# Hint: start with counts = {}, loop over rows, and use if/else
def count_values(column):
    counts = {}
    for row in rows:
        value = row[column]
        if value in counts:
            counts[value] = counts[value] + 1
        else:
            counts[value] = 1
    return counts

# Then count the Sex column
count_values("Sex")

Four of these values shouldn't be there. Can you spot them?

In [ ]:
# Your turn: count the Education level column
count_values("Education level")

Some values have stray spaces at the start or the end. Some are missing an apostrophe: `Bachelors` instead of `Bachelor's`. Part 2 fixes them.

## 3. A function that cleans a value

Three fixes tidy the Sex column.

- `.strip()` removes spaces at the ends.
- `.title()` fixes the capitals, so `"MALE"` becomes `"Male"`.
- A dictionary of known typos fixes the rest.

In [ ]:
fixes = {"Femal": "Female", "Mole": "Male"}

# Write clean_sex(value): strip, fix capitals with .title(), then, if the value is a key in fixes, return fixes[value]
def clean_sex(value):
    value = value.strip().title()
    if value in fixes:
        return fixes[value]
    return value

In [ ]:
# Test it on each messy value
for value in ["Femal", "female", "MALE", "Mole"]:
    print(value, "->", clean_sex(value))

In [ ]:
# Clean every row, then count Sex again
for row in rows:
    row["Sex"] = clean_sex(row["Sex"])

count_values("Sex")

## 4. Find an impossible age

ATUS ages run 15–85. The values 80 and 85 are top codes: they stand for older groups of people.

Ages are text too. `int()` turns text into a whole number, so `int('78')` gives the number `78`. You need it to compare an age with 85.

In [ ]:
# Print the Case and Age of anyone older than 85
for row in rows:
    if int(row["Age"]) > 85:
        print(row["Case"], row["Age"])

Someone is 150. You can't tell the real age from this file, so don't guess. Part 2 marks it as missing, and it checks the source file for another age problem.

That was a lot of looping for one column. Polars works on a whole column at once.

# Part 2: Polars

## 5. Load the data with Polars

Polars is a library for tables of data. If you haven't installed it yet:

- **In `dss-python`:** run `uv add polars` in a terminal. JupyterLab has one: **File → New → Terminal**. Then choose **Kernel → Restart Kernel…** so the notebook can find Polars. Run the Get the data cell again only if the files are missing. Part 2 doesn't use anything from Part 1, so you can start again from here.
- **With conda:** run `conda install -c conda-forge --override-channels polars`.
- **In Colab:** Polars is already there.

In [ ]:
import polars as pl

`pl.read_csv` reads the whole file into a DataFrame, which is a table. `null_values="NA"` tells Polars that the text `NA` means missing. Without it, the `Sleep (mins)` column would load as text.

In [ ]:
# Read atus_working.csv into df, with "NA" as missing, and show its shape
df = pl.read_csv("atus_working.csv", null_values="NA")
df.shape

In [ ]:
# Look at the first rows
df.head()

## 6. Fix the text columns

`with_columns` changes columns, and `pl.col("Sex")` picks one.

The `.str` methods are the same ideas as `clean_sex`, applied to every row at once. `.str.strip_chars()` is Polars' `.strip()`, and `.str.to_titlecase()` is its `.title()`. `replace` takes a dictionary, like `fixes`.

In [ ]:
# Clean Sex and Education level
df = df.with_columns(
    pl.col("Sex").str.strip_chars().str.to_titlecase().replace({"Femal": "Female", "Mole": "Male"}),
    pl.col("Education level").str.strip_chars().replace({"Bachelors": "Bachelor's", "Masters": "Master's"}),
)

In [ ]:
# Check Sex with value_counts()
df["Sex"].value_counts()

In [ ]:
# Your turn: check Education level the same way
df["Education level"].value_counts()

## 7. Codes, names and missing values

`rename` gives columns readable names. It takes a dictionary of old names and new names.

Metropolitan still uses ATUS codes, and a dictionary turns them into labels. `None` means missing. `replace_strict` is like `replace`, but the new values can be a different type: here, text for numbers.

In work hours, -1 (not asked) and -4 (hours vary) aren't hours, so they become missing too. So does the impossible age. `pl.when(...).then(...).otherwise(...)` is Polars' if/else for a column. `.alias("Age")` names the result, so it replaces the old column.

Empty is not zero: averages skip missing values. `df.null_count()` counts the missing values in each column.

In [ ]:
# Give TRCHILDNUM and TEHRUSLT readable names
df = df.rename({"TRCHILDNUM": "Number of children", "TEHRUSLT": "Work hours per week"})

In [ ]:
metro = {1: "Metropolitan", 2: "Non-metropolitan", 3: "Not identified", -1: None}

# Metropolitan codes to labels; negative work hours and ages over 85 become missing
df = df.with_columns(
    pl.col("Metropolitan").replace_strict(metro),
    pl.when(pl.col("Work hours per week") < 0).then(None).otherwise(pl.col("Work hours per week")).alias("Work hours per week"),
    pl.when(pl.col("Age") > 85).then(None).otherwise(pl.col("Age")).alias("Age"),
)

In [ ]:
# How many missing values does each column have now?
df.null_count()

## 8. Duplicates

Survey files can hold the same row twice. Check for it.

`df.is_duplicated()` marks each row that has an exact copy, and `.sum()` counts them. `filter` keeps the rows that match a condition, and `select` picks columns.

In [ ]:
# How many rows are exact copies of another row?
df.is_duplicated().sum()

10 means 5 rows, each there twice. `unique()` keeps one copy of each row, and `maintain_order=True` keeps the rows in their original order.

In [ ]:
# Keep one copy of each row, then show the new number of rows
df = df.unique(maintain_order=True)
df.height

In [ ]:
# Does any Case ID still appear twice?
df.filter(pl.col("Case").is_duplicated())

The same person appears twice, with two ages: 37 and 73. Don't guess which is right. Check the source file. In `atus_raw.csv`, `TUCASEID` is the case and `TEAGE` is the age.

In [ ]:
# Read atus_raw.csv and look up Case 20181109182318
raw = pl.read_csv("atus_raw.csv")
raw.filter(pl.col("TUCASEID") == 20181109182318).select("TUCASEID", "TEAGE")

The source says 37, so 73 is the wrong one. `~` means not and `&` means and, so the next cell keeps every row except the one with that Case and the age 73.

In [ ]:
# Drop the row with the wrong age, then check that every Case ID is unique
df = df.filter(~((pl.col("Case") == 20181109182318) & (pl.col("Age") == 73)))
df.height, df["Case"].n_unique()

## 9. Chores and leisure

- **Chores** = Interior cleaning + Laundry + Food and drink prep + Kitchen clean-up.
- **Leisure** = Reading + Playing games + TV and Movies + Socializing.

Eating and Grocery Shopping count as neither. These are the same definitions as the spreadsheets track.

`pl.sum_horizontal` adds up a list of columns, row by row. `.alias("Chores (mins)")` names the new column. Without it, the sum would overwrite the first column in the list.

In [ ]:
chores = [
    "Interior cleaning (mins)",
    "Laundry (mins)",
    "Food and drink prep (mins)",
    "Kitchen clean-up (mins)",
]
leisure = [
    "Reading (mins)",
    "Playing games (mins)",
    "TV and Movies (mins)",
    "Socializing (mins)",
]

# Add Chores (mins) and Leisure (mins): the sum of each list, row by row (pl.sum_horizontal), named with .alias
df = df.with_columns(
    pl.sum_horizontal(chores).alias("Chores (mins)"),
    pl.sum_horizontal(leisure).alias("Leisure (mins)"),
)

`select` picks columns or expressions. `.mean()` averages a column, and `.round(1)` keeps one decimal.

In [ ]:
# Your turn: the average Chores (mins) and Leisure (mins) over everyone, to one decimal
df.select(
    pl.col("Chores (mins)").mean().round(1),
    pl.col("Leisure (mins)").mean().round(1),
)

## 10. Answer the question

`group_by` makes one group for each value in a column, and `agg` summarizes each group. `pl.len()` counts the rows in a group, and `alias` names the new column. `clip(upper_bound=3)` puts 3 or more children in one group, so 3 means "3 or more".

`.sort("Children")` puts the groups in order; without it the order changes from run to run. `sort("People", descending=True)` puts the biggest group first.

In [ ]:
# Average chores and leisure by number of children (3 means 3 or more), in order of Children
by_kids = (
    df.with_columns(pl.col("Number of children").clip(upper_bound=3).alias("Children"))
    .group_by("Children")
    .agg(
        pl.len().alias("People"),
        pl.col("Chores (mins)").mean().round(1),
        pl.col("Leisure (mins)").mean().round(1),
    )
    .sort("Children")
)
by_kids

In [ ]:
# Your turn: the same by Labor force status, biggest group first
by_work = (
    df.group_by("Labor force status")
    .agg(
        pl.len().alias("People"),
        pl.col("Chores (mins)").mean().round(1),
        pl.col("Leisure (mins)").mean().round(1),
    )
    .sort("People", descending=True)
)
by_work

What you should see: more children means more chores and less leisure. People not in the labor force have the most leisure.

These are 2,000 people, not the US. The file has no survey weights, so the numbers describe this sample only.

## 11. One chart

You made a bar plot with matplotlib in the intro session. `plt.bar` takes the labels and the heights. `.to_list()` turns a Polars column into a list.

In [ ]:
import matplotlib.pyplot as plt

labels = ["0", "1", "2", "3 or more"]

# Bar chart of average chores by number of children
# Hint: the heights are by_kids["Chores (mins)"].to_list()
plt.bar(labels, by_kids["Chores (mins)"].to_list())
plt.xlabel("Children in the household")
plt.ylabel("Average chores (minutes per day)")
plt.title("More children, more chores")
plt.show()

## 12. Save your work

`write_csv` saves a DataFrame as a CSV file in this folder.

In [ ]:
# Save the clean data as atus_clean.csv
df.write_csv("atus_clean.csv")

## Check your numbers

| What | You should see |
|---|---|
| Rows at the start, after `unique()`, and after dropping the wrong age | 2006, then 2001, then 2000 |
| Sex values after cleaning | Female and Male |
| Education level values after cleaning | 8 |
| Missing values at the end of section 7 | Age 1, Sleep (mins) 3, Metropolitan 233, Work hours per week 839 |
| Average chores and leisure over everyone | 74.1 and 252.2 minutes |

Chores and leisure by number of children:

| Children | People | Chores (mins) | Leisure (mins) |
|---|---|---|---|
| 0 | 1137 | 65.7 | 297.9 |
| 1 | 323 | 66.2 | 206.9 |
| 2 | 360 | 85.9 | 187.7 |
| 3 or more | 180 | 117.4 | 173.5 |

Same numbers as the spreadsheets track? Compare at the wrap-up.

## Keep going

- Make the same chart for leisure.
- Group by `Diary Day` to compare weekends and weekdays.
- The [Exploratory Data Analysis with Polars workshop](https://colab.research.google.com/github/NCSU-Libraries/getting-started-with-python/blob/main/workshops/exploratory-data-analysis/exploratory-data-analysis.ipynb) goes further with the same library.
- The [Polars user guide](https://docs.pola.rs/) explains every method you used.
- [The American Time Use Survey](https://www.bls.gov/tus/) has the full data and documentation.
- Help: [go.ncsu.edu/getdatahelp](https://go.ncsu.edu/getdatahelp)